# Author: Geshna Garga
# Project: Customer Churn Analysis

# Customer Churn Analysis - Modeling

"""
Objective:
Build a predictive model to identify customers at risk of churn.

Steps:
1. Train-test split
2. Build baseline model
3. Evaluate performance
4. Identify key drivers
"""

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Geshna\OneDrive\Projects\Customer Churn Analysis\data\processed\cleaned_data.csv")

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [3]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [15]:
print("===== LOGISTIC REGRESSION =====")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc_score = roc_auc_score(y_test, y_prob)
print("\nROC-AUC Score:", roc_score)

===== LOGISTIC REGRESSION =====

Confusion Matrix:
[[915 118]
 [181 193]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407


ROC-AUC Score: 0.8319235288940887


The Logistic Regression model achieves an ROC-AUC score of 0.83, indicating strong discriminative ability. However, recall for churned customers is ~52%, meaning nearly half of at-risk customers may go undetected — highlighting the need for further optimization depending on business priorities

In [17]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.coef_[0]
})

feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))


Top 10 Important Features:
                           Feature  Importance
3                     TotalCharges    0.694885
10     InternetService_Fiber optic    0.650749
23             StreamingMovies_Yes    0.222440
21                 StreamingTV_Yes    0.212963
9                MultipleLines_Yes    0.160546
26            PaperlessBilling_Yes    0.134580
28  PaymentMethod_Electronic check    0.123558
0                    SeniorCitizen    0.094976
17            DeviceProtection_Yes    0.040129
8   MultipleLines_No phone service    0.029736


Customers with fiber optic internet and higher total charges show increased likelihood of churn, suggesting that premium service users may be more price-sensitive or have higher service expectations.

In [19]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_roc = roc_auc_score(y_test, y_prob_rf)

print("\n===== RANDOM FOREST =====")
print("ROC-AUC Score:", rf_roc)


===== RANDOM FOREST =====
ROC-AUC Score: 0.8160205206785698


Although Random Forest captures non-linear patterns, Logistic Regression outperforms it in this case, suggesting that churn behavior is largely driven by linear relationships such as pricing and contract structure.

In [21]:
df['churn_probability'] = model.predict_proba(scaler.transform(X))[:, 1]
high_risk = df[df['churn_probability'] > 0.7]
print("\nHigh-risk customers:", len(high_risk))


High-risk customers: 478


Approximately 7% of customers are classified as high-risk (churn probability > 0.7), representing a focused segment where targeted retention strategies can yield maximum ROI with minimal resource allocation.
Targeting high-risk customers (~478 users) could potentially prevent a monthly revenue loss of approximately ₹2,39,000, demonstrating significant business impact

Recommendation 1:

Offer discounts to customers on month-to-month contracts to encourage long-term commitment

Recommendation 2:

Focus retention efforts on customers with low tenure through onboarding programs

Recommendation 3:

Re-evaluate pricing strategies for high monthly charge segments

In [23]:
avg_revenue = 500  # ₹ per customer per month
potential_loss = len(high_risk) * avg_revenue

print("Estimated monthly revenue at risk: ₹", potential_loss)

Estimated monthly revenue at risk: ₹ 239000


"""
Model Performance:
- Logistic Regression ROC-AUC: ~0.83
- Random Forest ROC-AUC: ~0.81

Key Insights:
- High total charges and fiber optic users show higher churn risk
- Customers with short tenure are more likely to churn
- Pricing and contract structure strongly influence retention

Business Impact:
- ~7% customers identified as high-risk
- Potential revenue loss ≈ ₹2.3–2.5 lakh/month

Recommendations:
1. Target high-risk customers with retention campaigns
2. Promote long-term contracts via discounts
3. Improve onboarding for new customers

Future Improvements:
- Hyperparameter tuning
- Handling class imbalance
- Deployment using Streamlit
"""